# Barrier clip

### Add column to colonies GeoDataFrame showing if there is the presence of a barrier or not.

Outline of how to implement this:
* Import Delhi/NDMC/JJC dataset **[DONE]**
* Import barrier clip shapefiles **[DONE]**
* Reproject shapefiles to the same CRS **[DONE]**
* Check validity - turn into a function (that can be used for other shapefiles)
    * Make sure there are no duplicate rows **[DONE]**
    * Make sure there are no duplicate geometries **[DONE]**
    * Make sure that no row has None in geometry field. **[DONE]**
    * Check that barrier clip shapefiles are all (poly)lines. **[DONE]**
    * Make sure there are no validity problems. **[DONE]**
* Use Spatial Join to create identifier column `barrier`. **[DONE]**
    * Create function to do this
    * `barrier` column should be boolean (True or False)
    * Create intermediate columns 'canal', 'railway', and 'drain'.
    * Make 'barrier' as intersection of canal, railway and drain
* Export colonies shapefile. **[DONE]**
* Open in QGIS and manually inspect features.

In [ ]:
import os
from importlib import reload
import pandas as pd
import geopandas as gpd
import shapely
import spatial_index_utils

In [ ]:
reload(spatial_index_utils)

### Import Delhi/NDMC/JJC dataset

In [ ]:
colonies = gpd.read_file('delhi_ndmc_jjc_corrected.shp')

In [ ]:
colonies.head(2)

### Import barrier shapefiles

In [ ]:
barrier_clip_dir = 'Barrier_Clip'

canal_filepath = os.path.join(barrier_clip_dir, 'Canal', 'Canal.shp')
drain_filepath = os.path.join(barrier_clip_dir, 'Drain', 'Major_Drain.shp')
railway_filepath = os.path.join(barrier_clip_dir, 'Railway', 'Railway_Line.shp')

# Check filepath validity before importing shapefiles
print('canal filepath exists:', os.path.exists(canal_filepath))
print('drain filepath exists:', os.path.exists(drain_filepath))
print('railway filepath exists:', os.path.exists(railway_filepath))

In [ ]:
canal = gpd.read_file(canal_filepath)
drain = gpd.read_file(drain_filepath)
railway = gpd.read_file(railway_filepath)

In [ ]:
canal.head(2)

In [ ]:
drain.head(2)

In [ ]:
railway.head(2)

### Reproject shapefiles to EPSG:3857

In [ ]:
colonies = spatial_index_utils.reproject_gdf?

In [ ]:
colonies = spatial_index_utils.reproject_gdf(gdf=colonies, epsg_code=3857)

In [ ]:
canal = spatial_index_utils.reproject_gdf(gdf=canal, epsg_code=3857)

In [ ]:
drain = spatial_index_utils.reproject_gdf(gdf=drain, epsg_code=3857)

In [ ]:
railway = spatial_index_utils.reproject_gdf(gdf=railway, epsg_code=3857)

### Check validity of shapefiles (and correct)

#### First, check for duplicate rows

In [ ]:
gdf_list = {'colonies': colonies, 'canal': canal, 'drain': drain, 'railway': railway}

In [ ]:
for gdf in gdf_list:
    print(gdf, 'has duplicate rows:', spatial_index_utils.gdf_has_duplicate_rows(gdf_list[gdf]))

#### Second, correct any duplicate geometries

In [ ]:
len(colonies)

In [ ]:
colonies = spatial_index_utils.remove_duplicate_geom(colonies)

In [ ]:
len(colonies) # after removing duplicate geometries

In [ ]:
len(canal)

In [ ]:
canal = spatial_index_utils.remove_duplicate_geom(canal)

In [ ]:
len(canal)

In [ ]:
len(drain)

In [ ]:
drain = spatial_index_utils.remove_duplicate_geom(drain)

In [ ]:
len(drain)

In [ ]:
len(railway)

In [ ]:
railway = spatial_index_utils.remove_duplicate_geom(railway)

In [ ]:
len(railway)

#### Third, make sure no row has None in Geometry Field

In [ ]:
colonies[colonies['geometry'] == None]

In [ ]:
canal[canal['geometry'] == None]

In [ ]:
drain[drain['geometry'] == None]

In [ ]:
railway[railway['geometry'] == None]

#### Fourth, check that barrier clip shapefiles are all (poly)lines.

In [ ]:
geom_type = "Line"
for gdf in gdf_list:
    print(gdf, 'has all geometries of type', geom_type, ":",
          spatial_index_utils.check_geometries(gdf_list[gdf], geom_type))

#### Fifth, ensure no validity problems
* Canal has problems with rows 17 and 18.
* Railway has problems in multiple places

In [ ]:
canal.head()

In [ ]:
def print_invalid_rows(gdf):
    """Print rows with invalid geometries"""
    for i, row in gdf.iterrows():
        if not row['geometry'].is_valid:
            print('not valid index', i, '\n', row)
        

In [ ]:
# print_invalid_rows(canal) # 2 rows of invalid data

In [ ]:
print_invalid_rows(colonies)

In [ ]:
print_invalid_rows(drain)

In [ ]:
#print_invalid_rows(railway) #there were many rows

#### Test again with recorrected shapefiles (canal and railway)

In [ ]:
canal = gpd.read_file(canal_filepath)
canal = spatial_index_utils.reproject_gdf(gdf=canal, epsg_code=3857)
print_invalid_rows(canal)

In [ ]:
railway = gpd.read_file(railway_filepath)
railway = spatial_index_utils.reproject_gdf(gdf=railway, epsg_code=3857)

In [ ]:
print_invalid_rows(railway)

### Spatial Join for Intersections

let's start with colonies and canal

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, canal, "canal")

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, railway, "railway")

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, drain, "drain")

In [ ]:
colonies.head()

In [ ]:
# Create barrier column as being intersection with canal, railway or drain
colonies['barrier'] = colonies['canal'] | colonies['railway'] | colonies["drain"]

In [ ]:
colonies.head(10)

In [ ]:
colonies.tail(10)

### Save and export files

In [ ]:
colonies.to_file("colonies_with_barrier_3Aug2020.shp")

In [ ]:
with open("colonies_with_barrier_3Aug2020.data", "wb") as f:
    pickle.dump(colonies, f)